# 08 FAIR metadata with controlled access

**Deliverables:** A synthetic CSV, data dictionary, internal metadata, proposed discovery metadata, and FAIR gap report. Improve discoverability while reviewing metadata disclosure.

**Scenario:** Synthetic water-monitoring data for fictional Example River Nation. Roles and decisions are classroom examples.

**Objectives:** Explain the framework, implement a documented workflow, inspect a failed case, and revise the result. Each lab runs independently; lessons 00–06 provide background. Allow 60–90 minutes.

No real authorization, Label issuance, certification, or technical access enforcement results from running these cells.

In [1]:
LESSON = '08_fair_with_controlled_access'
import sys, json, csv
from pathlib import Path
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT/'toolkit'/'__init__.py').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError('Run from inside IDSov_toolkit')
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from toolkit.teaching import load_context, output_directory
# Applied labs explicitly use fictional data, even when nation.yaml exists.
ctx = load_context(REPO_ROOT, 'config/teaching_example.yaml')
out = output_directory(REPO_ROOT, ctx) / LESSON
out.mkdir(parents=True, exist_ok=True)
def save_json(name, obj):
    (out / name).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')
def save_csv(name, rows):
    with (out/name).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
print('Fictional teaching scenario:', ctx.config.name)
print('Draft artifacts:', out)


Fictional teaching scenario: Example River Nation (fictional)
Draft artifacts: C:\Users\gekek\Documents\IDSov_toolkit\outputs\teaching\08_fair_with_controlled_access


## Describe data so another analyst can interpret it

FAIR concerns both data and metadata. Persistent identifiers and searchable registration require services and stewardship beyond a local JSON file. Authentication is compatible with FAIR (A1.2); accessibility does not require unrestricted release.


In [2]:
rows = [{'period':'2024-01','zone':'zone-A','water_level_m':2.4}, {'period':'2024-02','zone':'zone-A','water_level_m':2.1}]
save_csv('synthetic_water.csv', rows)
dictionary = [dict(field='period', type='string', unit='calendar month', definition='Synthetic observation month', missing='empty'), dict(field='zone', type='string', unit='none', definition='Fictional aggregation identifier; no real geography', missing='not allowed'), dict(field='water_level_m', type='number', unit='m', definition='Synthetic height above fictional local datum', missing='empty')]
save_csv('data_dictionary.csv', dictionary)
metadata = {'identifier':'urn:example:idsov:water:v1', 'identifier_status':'local teaching identifier; persistence not established', 'title':'Synthetic monthly water levels', 'version':'1.0', 'format':'text/csv', 'encoding':'UTF-8', 'classification':'RESTRICTED', 'access_protocol':'Proposed HTTPS service with authentication and steward authorization; not deployed', 'access_request':'Fictional steward review workflow', 'reuse_terms':'No reuse until scoped authorization; no open license assigned', 'provenance':'Entirely invented classroom observations', 'dictionary':'data_dictionary.csv', 'metadata_retention':'Retain a reviewed tombstone if data are withdrawn', 'sensitive_internal_fields':['sampling_location','steward_email'], 'sampling_location':'Fictional internal location descriptor', 'steward_email':'steward@example.invalid'}
save_json('internal_metadata.json', metadata)


## Review discovery metadata independently

Even a title, coverage description, or contact can disclose sensitive information. Build an allowlisted discovery candidate. A field filter is a teaching mechanism; human disclosure review is still required. A use restriction is not automatically a complete license or rights statement.


In [3]:
allowed = ['identifier','identifier_status','title','version','format','encoding','access_request','reuse_terms','provenance']
discovery = {k:metadata[k] for k in allowed}
discovery['disclosure_review_status'] = 'pending; not published'
assert 'sampling_location' not in discovery and 'steward_email' not in discovery
save_json('discovery_metadata_candidate.json', discovery)
checks = [
    {'principle':'F1','implemented':'Local identifier with version','gap':'Choose persistent identifier service and maintenance owner'},
    {'principle':'F2/F3','implemented':'Description includes dataset identifier','gap':'Review richness and disclosure with steward'},
    {'principle':'F4','implemented':'Local discovery candidate','gap':'No searchable catalog registration'},
    {'principle':'A1/A1.1/A1.2','implemented':'Documented proposed controlled access','gap':'Deploy and test retrieval and authorization'},
    {'principle':'A2','implemented':'Tombstone retention proposal','gap':'Assign repository retention responsibility'},
    {'principle':'I1/I2/I3','implemented':'CSV, dictionary, explicit provenance description','gap':'Select shared vocabularies and qualified linked metadata'},
    {'principle':'R1/R1.1/R1.2/R1.3','implemented':'Units, purpose, provenance, proposed restrictions','gap':'Resolve rights statement and domain standards with reviewers'},
]
save_csv('fair_gap_report.csv', checks)
for check in checks: print(check['principle'], check['gap'])


F1 Choose persistent identifier service and maintenance owner
F2/F3 Review richness and disclosure with steward
F4 No searchable catalog registration
A1/A1.1/A1.2 Deploy and test retrieval and authorization
A2 Assign repository retention responsibility
I1/I2/I3 Select shared vocabularies and qualified linked metadata
R1/R1.1/R1.2/R1.3 Resolve rights statement and domain standards with reviewers


## Exercise a withdrawal and reproducibility check

Read the serialized CSV rather than trusting the in-memory values. Then preserve reviewed discovery information when observations become unavailable. The tombstone remains a draft; metadata retention must follow the governing policy.


In [4]:
with (out/'synthetic_water.csv').open(encoding='utf-8', newline='') as f:
    loaded = list(csv.DictReader(f))
mean_level = sum(float(r['water_level_m']) for r in loaded)/len(loaded)
assert abs(mean_level-2.25)<1e-10
print('Synthetic mean:', mean_level, 'm; invented local datum')
tombstone = dict(discovery, data_status='withdrawn in classroom simulation', access_request='Contact fictional steward for reviewed explanation', disclosure_review_status='pending')
save_json('metadata_tombstone_candidate.json', tombstone)


Synthetic mean: 2.25 m; invented local datum


## Independent task

Add a third observation with a missing value and define an explicit missingness rule. Revise the dictionary and calculation. Choose a domain vocabulary and document its URI, version, and fit. Explain why these local files do not establish full FAIR implementation.

## Completion checklist

- [ ] CSV round-trip and units are checked.
- [ ] Discovery candidate excludes internal fields and remains pending.
- [ ] All four FAIR dimensions have implementation evidence and gaps.
- [ ] Withdrawal preserves a reviewed metadata plan.

## Primary references

- [FAIR Guiding Principles](https://www.gofair.foundation/fair-principles)